In [1]:
from bs4 import BeautifulSoup
import requests, os, csv, urllib.parse, urllib3, time, random
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry

import numpy as np
import pandas as pd
from tqdm import tqdm
import re,os,glob, time,string
urllib3.disable_warnings()

import web_general
from IPython.display import Image

import requests
from bs4 import BeautifulSoup
import time

In [2]:
base_url = 'https://www.beagledatabase.co.za/supplier_search/new'

In [3]:
soup=web_general.parse_listing(web_general.buildSearchUrl(base_url), web_general.get_proxies())
if not soup:print("Not found")
else: print("good to go")

good to go


In [4]:
soup

<!DOCTYPE html>
<html>
<head>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta charset="utf-8"/>
<meta content="ie=edge" http-equiv="x-ua-compatible"/>
<meta content="width=device-width, initial-scale=1.0" name="viewport"/>
<title>Mpowered Beagle Database</title>
<link href="/assets/modern/base-1f1ef6ff8ac6d1528c7f6f8dc40da98144559e5b1595986e5189abd26979a98a.css" media="screen" rel="stylesheet"/>
<link href="/assets/modern/default-e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855.css" media="screen" rel="stylesheet"/>
<meta content="authenticity_token" name="csrf-param"/>
<meta content="j4KfXZIzDuXnQ9Ety+sy2cqVD46vfw8rjqXYdezfrBvlDNyisp4uRKn3MiyTR84wmqSaCTyztBzp9ZY/mm2qGw==" name="csrf-token"/>
<script src="https://mpowered.atlassian.net/s/d41d8cd98f00b204e9800998ecf8427e-T/-egccmf/b/24/a44af77267a987a660377e5c46e0fb64/_/download/batch/com.atlassian.jira.collector.plugin.jira-issue-collector-plugin:issuecollector/com.atlassian.jira.collector.plu

In [5]:
URL='https://www.beagledatabase.co.za/\
supplier_search/results?utf8=%E2%9C%93&supplier_search%5Bkeyword%5D=\
&supplier_search%5Blevel%5D=&supplier_search%5Bblack_ownership%5D=&\
supplier_search%5Bblack_female_ownership%5D=&supplier_search%5Bcompany_size%5D=&supplier_search%5Bregions%5D=&supplier_search%5Bcommodity%5D=Automobiles+%26+Parts&commit=Search'

In [6]:
soup=web_general.parse_listing(web_general.buildSearchUrl(URL), web_general.get_proxies())
if not soup:print("Not found")
else: print("good to go")

good to go


### URL BUILDER FUNCTION

In [7]:
def buildSearchUrl(commodity_name, region, company_size, page):
    base_url = "https://www.beagledatabase.co.za/supplier_search/results"
    params = {
    "commit": "Search",
    "page": str(page),
    "supplier_search[black_female_ownership]": "",
    "supplier_search[black_ownership]": "",
    "supplier_search[commodity]": commodity_name,
    "supplier_search[company_size]": company_size,
    "supplier_search[keyword]": "",
    "supplier_search[level]": "",
    "supplier_search[regions]": region,
    "utf8": "✓"
    }
    encoded_params = urllib.parse.urlencode(params)
    encoded_params = encoded_params.replace("%2B%25","+%").replace("%2B","+")
    url = f"{base_url}?{encoded_params}"
    return url

In [8]:
buildSearchUrl("Automobiles+%26+Parts",2,2,1)

'https://www.beagledatabase.co.za/supplier_search/results?commit=Search&page=1&supplier_search%5Bblack_female_ownership%5D=&supplier_search%5Bblack_ownership%5D=&supplier_search%5Bcommodity%5D=Automobiles+%26+Parts&supplier_search%5Bcompany_size%5D=2&supplier_search%5Bkeyword%5D=&supplier_search%5Blevel%5D=&supplier_search%5Bregions%5D=2&utf8=%E2%9C%93'

In [9]:
buildSearchUrl("Automobiles+%26+Parts",2,3,2)

'https://www.beagledatabase.co.za/supplier_search/results?commit=Search&page=2&supplier_search%5Bblack_female_ownership%5D=&supplier_search%5Bblack_ownership%5D=&supplier_search%5Bcommodity%5D=Automobiles+%26+Parts&supplier_search%5Bcompany_size%5D=3&supplier_search%5Bkeyword%5D=&supplier_search%5Blevel%5D=&supplier_search%5Bregions%5D=2&utf8=%E2%9C%93'

In [10]:
buildSearchUrl("Automobiles+%26+Parts",10,2,2)

'https://www.beagledatabase.co.za/supplier_search/results?commit=Search&page=2&supplier_search%5Bblack_female_ownership%5D=&supplier_search%5Bblack_ownership%5D=&supplier_search%5Bcommodity%5D=Automobiles+%26+Parts&supplier_search%5Bcompany_size%5D=2&supplier_search%5Bkeyword%5D=&supplier_search%5Blevel%5D=&supplier_search%5Bregions%5D=10&utf8=%E2%9C%93'

In [11]:
commodities = ["Automobiles+%26+Parts","Forestry+%26+Paper","Industrial+Metals","Mining","Construction+%26+Materials"
,"Equity+Investment+Instruments","Nonequity+Investment+Instruments","Insurance+Services",
"Financial+Management+Services","Banking+Services","Beverages","Food+Producers",
"Health+Care+Equipment+%26+Services","Pharmaceuticals+%26+Biotechnology","Staffing+solutions",
"HR+Services","Training+Provider","Aerospace+%26+Defense","General+Industrials",
"Electronic+%26+Electrical+Equipment","Industrial+Engineering","Industrial+Transportation+%26+Warehousing"
,"Media","Fuels","Gases","Chemicals","Household+Goods+%26+Home+Construction","Leisure+Goods",
"Personal+Goods","Tobacco","Consulting","Legal+Services","Auditing","Real+Estate+Investments"
,"Real+Estate+Services","Food+%26+Drug+Retailers","General+Retailers","Marketing",
"Travel+%26+Accommodation","Stationery","Document+Management","Conferences+%26+Events",
"Telecommunications","Software+%26+Computer+Services","Technology+Hardware+%26+Equipment",
"Electricity","Gas%2C+Water+%26+Multiutilities","Other"]

region_dict = {2:'Eastern Cape', 3:'Free State',4:'Gauteng', 5:'KwaZulu-Natal', 6:'Limpopo', 
          7:'Mpumalanga', 1:'National', 9:'North West',8:'Northern Cape', 10:'Western Cape'}

company_size_dict = {1:'Exempt Micro Enterprise', 2:'Qualifying Small Enterprise', 3:'Generic'}

In [12]:
dataframes = []
for commodity in commodities:
    for region, region_name in region_dict.items():
        for company_size,company_size_name in company_size_dict.items():
            commodity_name = commodity.replace("+%26+", " & ").replace("%2C+", ", ").replace("+"," ")
            print("################################################################################")
            print("################################################################################")
            print(f"building url for commodity: \033[1;34m{commodity_name}\033[0m with region: \033[1;33m{region_name}\033[0m \nand company size \033[1;35m{company_size_name}\033[0m")
            url = buildSearchUrl(commodity, region, company_size, 1)
            soup = web_general.parse_listing(url, web_general.get_proxies())
            if soup:
                print(f"extracting table from soup for page 1")
                table = soup.find('table', class_='company_table supplier_search')
                if table:
                    print(f"creating dataframe for page 1")
                    df = pd.read_html(str(table))[0]
                    df['commodity'] = commodity_name
                    df["Region"] = region_name
                    df['Company_size'] = company_size_name
                    dataframes.append(df)
                    
                    print(f"checking if they're other pages")
                    pagination = soup.find("div", class_="apple_pagination")
                    if pagination:
                        print("other pages were found")
                        pages = pagination.find_all("a")
                        for page in range(2, len(pages) + 1):
                            url = buildSearchUrl(commodity, region, company_size, page)
                            soup = web_general.parse_listing(url, web_general.get_proxies())
                            if soup:
                                print(f"extracting table from soup for Page {page}")
                                table = soup.find('table', class_='company_table supplier_search')
                                if table:
                                    print(f"creating dataframe  for Page {page}")
                                    df = pd.read_html(str(table))[0]
                                    df['commodity'] = commodity_name
                                    df["Region"] = region_name
                                    df['Company_size'] = company_size_name

                                    dataframes.append(df)
                                else:
                                    print(f"No table found for commodity {commodity_name} - Page {page}")
                            else:
                                print(f"Failed to fetch data for commodity {commodity_name} - Page {page}")
                    else:
                        print(f"No other pages were found for commodity: {commodity_name} with region: {region_name} \nand company size {company_size_name}")
                    print(f"Done with commodity: {commodity_name} with region: {region_name} \nand company size {company_size_name}")   
                else:
                    print(f"No table found for commodity {commodity_name}")
            else:
                print(f"Failed to fetch data for commodity {commodity_name}")

# Concatenate all dataframes into a single dataframe
result_df = pd.concat(dataframes, ignore_index=True)
result_df.drop("Unnamed: 5", axis=1, inplace=True)

################################################################################
################################################################################
building url for commodity: Automobiles & Parts with region: Eastern Cape 
and company size Exempt Micro Enterprise
extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Automobiles & Parts with region: Eastern Cape 
and company size Exempt Micro Enterprise
Done with commodity: Automobiles & Parts with region: Eastern Cape 
and company size Exempt Micro Enterprise
################################################################################
################################################################################
building url for commodity: Automobiles & Parts with region: Eastern Cape 
and company size Qualifying Small Enterprise
extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No

extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Automobiles & Parts with region: Limpopo 
and company size Qualifying Small Enterprise
Done with commodity: Automobiles & Parts with region: Limpopo 
and company size Qualifying Small Enterprise
################################################################################
################################################################################
building url for commodity: Automobiles & Parts with region: Limpopo 
and company size Generic
extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Automobiles & Parts with region: Limpopo 
and company size Generic
Done with commodity: Automobiles & Parts with region: Limpopo 
and company size Generic
################################################################################
##############################

extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Automobiles & Parts with region: Northern Cape 
and company size Generic
Done with commodity: Automobiles & Parts with region: Northern Cape 
and company size Generic
################################################################################
################################################################################
building url for commodity: Automobiles & Parts with region: Western Cape 
and company size Exempt Micro Enterprise
extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Automobiles & Parts with region: Western Cape 
and company size Exempt Micro Enterprise
Done with commodity: Automobiles & Parts with region: Western Cape 
and company size Exempt Micro Enterprise
############################################################################

extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Forestry & Paper with region: KwaZulu-Natal 
and company size Qualifying Small Enterprise
Done with commodity: Forestry & Paper with region: KwaZulu-Natal 
and company size Qualifying Small Enterprise
################################################################################
################################################################################
building url for commodity: Forestry & Paper with region: KwaZulu-Natal 
and company size Generic
extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Forestry & Paper with region: KwaZulu-Natal 
and company size Generic
Done with commodity: Forestry & Paper with region: KwaZulu-Natal 
and company size Generic
################################################################################
###############

extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Forestry & Paper with region: Northern Cape 
and company size Exempt Micro Enterprise
Done with commodity: Forestry & Paper with region: Northern Cape 
and company size Exempt Micro Enterprise
################################################################################
################################################################################
building url for commodity: Forestry & Paper with region: Northern Cape 
and company size Qualifying Small Enterprise
extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Forestry & Paper with region: Northern Cape 
and company size Qualifying Small Enterprise
Done with commodity: Forestry & Paper with region: Northern Cape 
and company size Qualifying Small Enterprise
############################################

extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Industrial Metals with region: Gauteng 
and company size Generic
Done with commodity: Industrial Metals with region: Gauteng 
and company size Generic
################################################################################
################################################################################
building url for commodity: Industrial Metals with region: KwaZulu-Natal 
and company size Exempt Micro Enterprise
extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Industrial Metals with region: KwaZulu-Natal 
and company size Exempt Micro Enterprise
Done with commodity: Industrial Metals with region: KwaZulu-Natal 
and company size Exempt Micro Enterprise
################################################################################
##############

... (97 outputs left out)

extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Gas, Water & Multiutilities with region: Limpopo 
and company size Qualifying Small Enterprise
Done with commodity: Gas, Water & Multiutilities with region: Limpopo 
and company size Qualifying Small Enterprise
################################################################################
################################################################################
building url for commodity: Gas, Water & Multiutilities with region: Limpopo 
and company size Generic
extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Gas, Water & Multiutilities with region: Limpopo 
and company size Generic
Done with commodity: Gas, Water & Multiutilities with region: Limpopo 
and company size Generic
#######################################################################

extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Gas, Water & Multiutilities with region: Northern Cape 
and company size Generic
Done with commodity: Gas, Water & Multiutilities with region: Northern Cape 
and company size Generic
################################################################################
################################################################################
building url for commodity: Gas, Water & Multiutilities with region: Western Cape 
and company size Exempt Micro Enterprise
extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Gas, Water & Multiutilities with region: Western Cape 
and company size Exempt Micro Enterprise
Done with commodity: Gas, Water & Multiutilities with region: Western Cape 
and company size Exempt Micro Enterprise
####################################

extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
other pages were found
extracting table from soup for Page 2
creating dataframe  for Page 2
extracting table from soup for Page 3
creating dataframe  for Page 3
extracting table from soup for Page 4
creating dataframe  for Page 4
extracting table from soup for Page 5
creating dataframe  for Page 5
extracting table from soup for Page 6
creating dataframe  for Page 6
extracting table from soup for Page 7
creating dataframe  for Page 7
extracting table from soup for Page 8
creating dataframe  for Page 8
extracting table from soup for Page 9
creating dataframe  for Page 9
extracting table from soup for Page 10
creating dataframe  for Page 10
Done with commodity: Other with region: Gauteng 
and company size Generic
################################################################################
################################################################################
building url for c

extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Other with region: North West 
and company size Exempt Micro Enterprise
Done with commodity: Other with region: North West 
and company size Exempt Micro Enterprise
################################################################################
################################################################################
building url for commodity: Other with region: North West 
and company size Qualifying Small Enterprise
extracting table from soup for page 1
creating dataframe for page 1
checking if they're other pages
No other pages were found for commodity: Other with region: North West 
and company size Qualifying Small Enterprise
Done with commodity: Other with region: North West 
and company size Qualifying Small Enterprise
################################################################################
#################################

In [13]:
result_df

,Company Trading Name,Vat Number,Latest BO,Latest BFO,Latest Level,commodity,Region,Company_size
0,Ralph's Body Shop CC t/a AUTOTECH PANELBEATERS,4510153184,0.00,0.00,4,Automobiles & Parts,Eastern Cape,Exempt Micro Enterprise
1,5 Mahem Place Properties CC t/a Elliot Panelbe...,4620205445,60.00,60.00,2,Automobiles & Parts,Eastern Cape,Exempt Micro Enterprise
2,A.C Koekemoer T/A Econo Body Works,4850260391,0.00,0.00,4,Automobiles & Parts,Eastern Cape,Exempt Micro Enterprise
3,Alced Touch Ups cc T/A Dent Doctor,4360179156,100.00,0.00,1,Automobiles & Parts,Eastern Cape,Exempt Micro Enterprise
4,ALCO TRUCK INVESTMENTS 34 CC T/A ALCO TRUCK AN...,4440262311,0.00,0.00,4,Automobiles & Parts,Eastern Cape,Exempt Micro Enterprise
...,...,...,...,...,...,...,...,...
62358,COMPUGROUP MEDICAL SOUTH AFRICA (PTY) LTD,4320226063,0.00,0.00,Non-compliant,Other,Western Cape,Generic
62359,Truworths Ltd,4630103754,15.32,8.05,6,Other,Western Cape,Generic
62360,Credit Collections Corporation (Pty) Ltd,4580293886,15.32,8.05,6,Other,Western Cape,Generic
62361,K2020211444 (SOUTH AFRICA) (Pty) Ltd,4050291808,15.32,8.05,6,Other,Western Cape,Generic


In [14]:
result_df.shape

(62363, 8)

In [15]:
result_df.to_csv("Mpowered_base.csv")